# EMEA AI Cohort — Assignment 6
## MLP for Heart Disease Prediction

**Goal:** Train a Multi-Layer Perceptron (MLP) neural network to predict whether a patient has heart disease using clinical measurements.

### What this notebook does
1. Loads the UCI Heart Disease (Cleveland) dataset.
2. Cleans missing values and converts the original target into a binary classification label.
3. Explores and prepares the data.
4. Splits the data into training and test sets.
5. Trains an MLP using a preprocessing pipeline.
6. Evaluates it with accuracy, precision, recall, F1-score, ROC-AUC, a classification report, and a confusion matrix.
7. Plots the training loss and ROC curve, then summarizes the results generated by the run.

**Important:** The dataset contains clinical information and this is an educational ML exercise, not a medical diagnostic tool. The notebook does not hard-code or invent performance scores; metrics are calculated from the test set when you run it.

## 1. Install / import libraries

The notebook uses common Python data-science libraries. `scikit-learn` provides the MLP model and evaluation metrics; `pandas` and `matplotlib` handle the data and plots.

In [ ]:
# If a library is missing in your environment, uncomment and run:
# %pip install pandas numpy scikit-learn matplotlib

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

## 2. Load the dataset

This notebook uses the **processed Cleveland subset** of the UCI Heart Disease dataset. It has 13 input features and a `num` target column. In the original data, `num = 0` means no diagnosed heart disease, while values `1` through `4` indicate presence/severity. For this binary task, we map `0 → 0` and `1–4 → 1`.

The dataset is downloaded from the UCI repository, so an internet connection is needed for the first load.

In [ ]:
data_url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "heart-disease/processed.cleveland.data"
)

column_names = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"
]

try:
    df = pd.read_csv(
        data_url,
        header=None,
        names=column_names,
        na_values="?",
        skipinitialspace=True
    )
except Exception as exc:
    raise RuntimeError(
        "Could not download the UCI Cleveland Heart Disease dataset. "
        "Check your internet connection and try again. Original error: "
        + str(exc)
    )

# Convert columns that may contain '?' missing values to numeric.
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Binary target: 0 = no heart disease, 1 = heart disease present.
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])

print("Dataset loaded.")
print("Shape:", df.shape)
display(df.head())

## 3. Inspect and clean the data

We inspect column types, missing values, and the target balance. Missing feature values are imputed inside the model pipeline using the median. Doing imputation inside the pipeline is important because the imputer is fitted only on the training data, helping prevent data leakage.

In [ ]:
print("Dataset information:")
df.info()

print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\nTarget counts (0 = no disease, 1 = disease present):")
display(df["target"].value_counts().sort_index().to_frame("count"))

print("\nTarget proportions:")
display(df["target"].value_counts(normalize=True).sort_index().round(3).to_frame("proportion"))

In [ ]:
# Plot the class distribution so class imbalance is visible.
target_counts = df["target"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(["No disease (0)", "Disease present (1)"], target_counts.values)
plt.title("Target Class Distribution")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

## 4. Separate features and target; create train/test sets

The test set is held back until evaluation. `stratify=y` preserves approximately the same class proportions in both sets. We use a 80/20 split and a fixed random seed so the split is reproducible.

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)
print("Training target distribution:")
display(y_train.value_counts(normalize=True).sort_index().round(3).to_frame("proportion"))
print("Test target distribution:")
display(y_test.value_counts(normalize=True).sort_index().round(3).to_frame("proportion"))

## 5. Build and train the MLP model

An MLP is a feed-forward neural network made of an input layer, one or more hidden layers, and an output layer. We use two hidden layers `(32, 16)` and ReLU activation. Because neural networks are sensitive to feature scales, standardization is included in the pipeline.

- **Median imputation:** handles missing clinical values.
- **StandardScaler:** puts numeric features on comparable scales.
- **MLPClassifier:** learns non-linear relationships.
- **Early stopping:** reserves part of the training data for validation and stops training when validation performance no longer improves.

The architecture and settings are a baseline, not a claim that this is the best possible model.

In [ ]:
mlp_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=25,
        random_state=RANDOM_STATE
    ))
])

mlp_pipeline.fit(X_train, y_train)
mlp = mlp_pipeline.named_steps["mlp"]

print("MLP training complete.")
print("Iterations run:", mlp.n_iter_)
print("Final training loss:", round(float(mlp.loss_), 6))
print("Number of input features:", X_train.shape[1])

## 6. Evaluate on the held-out test set

We calculate all metrics from predictions on the test set. For this medical classification example, precision and recall matter in addition to accuracy. Recall for class 1 measures the fraction of patients in the test set who have disease and are correctly identified by the model. This still does **not** establish clinical usefulness.

In [ ]:
y_pred = mlp_pipeline.predict(X_test)
y_prob = mlp_pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision (positive class)": precision_score(y_test, y_pred, zero_division=0),
    "Recall (positive class)": recall_score(y_test, y_pred, zero_division=0),
    "F1-score (positive class)": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

metrics_df = pd.DataFrame({
    "Metric": list(metrics.keys()),
    "Test-set result": list(metrics.values())
})
metrics_df["Test-set result"] = metrics_df["Test-set result"].round(4)
display(metrics_df)

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["No disease", "Disease present"],
    zero_division=0
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No disease", "Disease present"]
)
disp.plot(values_format="d")
plt.title("MLP Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

print("Confusion matrix layout:")
print("[[true negatives, false positives],")
print(" [false negatives, true positives]]")

In [ ]:
# ROC curve using predicted probabilities
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve — MLP on Test Set")
plt.tight_layout()
plt.show()

In [ ]:
# Training loss curve: loss at each optimization iteration
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(mlp.loss_curve_) + 1), mlp.loss_curve_)
plt.xlabel("Iteration")
plt.ylabel("Training loss")
plt.title("MLP Training Loss Curve")
plt.tight_layout()
plt.show()

## 7. Print a run-specific results summary

This cell creates a short summary using the actual metrics calculated above. It deliberately does not include pre-filled performance values. Interpret results cautiously because the dataset is relatively small and this single split can produce variable scores.

In [ ]:
print("RESULTS SUMMARY (generated from this notebook run)")
print("-" * 55)
print(f"Rows used: {len(df)}")
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"MLP hidden layers: {mlp.hidden_layer_sizes}")
print(f"Training iterations: {mlp.n_iter_}")
print(f"Final training loss: {mlp.loss_:.4f}")
print()
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

print("\nInterpretation guide:")
print("- Accuracy: overall fraction of test predictions that were correct.")
print("- Precision: among predicted positive cases, fraction actually positive.")
print("- Recall: among actual positive cases, fraction detected by the model.")
print("- F1-score: harmonic mean of precision and recall.")
print("- ROC-AUC: ability to rank positive cases above negative cases across thresholds.")
print("\nUse the values printed above as your actual experimental results for this run.")

## 8. Conclusion

After running all cells, write your conclusion based on the metrics and plots produced in your environment. Consider answering:

1. What accuracy, precision, recall, F1-score, and ROC-AUC did the MLP obtain?
2. From the confusion matrix, how many false negatives were there?
3. Did the loss curve generally decrease?
4. What limitations should be acknowledged?

### Example conclusion template — fill with your actual outputs
> An MLP classifier was trained on the UCI Cleveland Heart Disease dataset using median imputation and feature standardization. The held-out test-set accuracy was **[copy actual value]**, the positive-class recall was **[copy actual value]**, and ROC-AUC was **[copy actual value]**. The confusion matrix showed **[copy false-negative count]** false negatives. These results describe only this train/test split; the dataset is small, and the model must not be used for medical decisions.

## 9. Key concepts learned

- **MLP:** a neural network that can model non-linear relationships.
- **Feature scaling:** helps neural-network optimization.
- **Train/test split:** evaluates performance on held-out data.
- **Precision and recall:** provide different views of classification errors.
- **ROC-AUC:** summarizes ranking performance over classification thresholds.
- **Data leakage prevention:** preprocessing is fitted within a pipeline on training data only.

### References
- UCI Machine Learning Repository, Heart Disease dataset: https://archive.ics.uci.edu/dataset/45/heart+disease
- scikit-learn MLPClassifier documentation: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html
- scikit-learn model evaluation: https://scikit-learn.org/stable/modules/model_evaluation.html